# ChessMind Arena — ResNet Policy Model Training

This notebook trains an improved chess policy network for ChessMind Arena.

The model learns to rank promising legal chess moves for a given position.

Training data:
- Chess positions from the ChessMind Arena Policy Dataset V2
- Stockfish-ranked candidate moves
- Multiple candidate moves per position

Architecture:
- 18-channel chessboard representation
- Residual CNN backbone
- Policy head with 20,480 move classes
- Legal-move masking
- Rank-weighted candidate supervision

The policy network is intended to work independently first.
Later, we will compare:

1. Policy-only move selection
2. Policy + LLM move selection

Stockfish is used as the source of training labels and for offline evaluation,
not as the move selector during experimental gameplay.

In [1]:
!pip install -q datasets python-chess torch pandas numpy tqdm scikit-learn huggingface_hub

In [2]:
import os
import json
import random
import getpass

from pathlib import Path
from copy import deepcopy

import chess
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

from datasets import load_dataset
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

from huggingface_hub import HfApi

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.14.1+cu130
CUDA available: True
GPU: Tesla T4


In [3]:
SEED = 42

HF_DATASET_ID = "Mridul-Dixit/chessmind-arena-policy-v2"
HF_MODEL_ID = "Mridul-Dixit/chessmind-arena-policy-model"

BATCH_SIZE = 128
EPOCHS = 20

LEARNING_RATE = 3e-4
WEIGHT_DECAY = 1e-4

PATIENCE = 4
MIN_DELTA = 0.0005

NUM_WORKERS = 0

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

HF_TOKEN = getpass.getpass("Enter your Hugging Face token: ")

os.environ["HF_TOKEN"] = HF_TOKEN

print("Device:", DEVICE)

Device: cuda


In [4]:
dataset = load_dataset(
    HF_DATASET_ID,
    token=HF_TOKEN
)

print(dataset)

Using the latest cached version of the dataset since Mridul-Dixit/chessmind-arena-policy-v2 couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'default' at /root/.cache/huggingface/datasets/Mridul-Dixit___chessmind-arena-policy-v2/default/0.0.0/7f6c5e242584a1a20e1843788dc50e84be5f5fe2 (last modified on Sun Nov 22 08:01:36 2026).


DatasetDict({
    train: Dataset({
        features: ['fen', 'target_move', 'candidates', 'nnue_evaluations', 'game_id', 'ply', 'result', 'source_split', 'nodes', 'multi_pv', 'opening_multi_pv', 'opening_plies', 'stockfish_version'],
        num_rows: 80000
    })
    validation: Dataset({
        features: ['fen', 'target_move', 'candidates', 'nnue_evaluations', 'game_id', 'ply', 'result', 'source_split', 'nodes', 'multi_pv', 'opening_multi_pv', 'opening_plies', 'stockfish_version'],
        num_rows: 10000
    })
    test: Dataset({
        features: ['fen', 'target_move', 'candidates', 'nnue_evaluations', 'game_id', 'ply', 'result', 'source_split', 'nodes', 'multi_pv', 'opening_multi_pv', 'opening_plies', 'stockfish_version'],
        num_rows: 10000
    })
})


In [5]:
required_splits = {"train", "validation", "test"}

if not required_splits.issubset(set(dataset.keys())):
    raise ValueError(
        f"Expected splits {required_splits}, "
        f"but found {list(dataset.keys())}"
    )

required_columns = {
    "fen",
    "target_move",
    "candidates"
}

for split_name in required_splits:
    missing = required_columns - set(dataset[split_name].column_names)

    if missing:
        raise ValueError(
            f"{split_name} is missing columns: {missing}"
        )

# Keep only the fields required for policy training.
# In particular, we do not need to load nnue_evaluations into each batch.
dataset = dataset.select_columns([
    "fen",
    "target_move",
    "candidates"
])

print("\nDataset sizes:")
for split_name in ["train", "validation", "test"]:
    print(split_name, len(dataset[split_name]))

sample = dataset["train"][0]

print("\nSample FEN:", sample["fen"])
print("Target move:", sample["target_move"])
print("Candidates:", sample["candidates"][:5])


Dataset sizes:
train 80000
validation 10000
test 10000

Sample FEN: rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w KQkq - 0 1
Target move: e2e4
Candidates: [{'move': 'e2e4', 'rank': 1, 'score_cp': 18}, {'move': 'c2c4', 'rank': 2, 'score_cp': 1}, {'move': 'd2d3', 'rank': 3, 'score_cp': -2}, {'move': 'a2a3', 'rank': 4, 'score_cp': -3}, {'move': 'a2a4', 'rank': 5, 'score_cp': -7}]


In [6]:
PIECE_TO_CHANNEL = {
    (chess.WHITE, chess.PAWN): 0,
    (chess.WHITE, chess.KNIGHT): 1,
    (chess.WHITE, chess.BISHOP): 2,
    (chess.WHITE, chess.ROOK): 3,
    (chess.WHITE, chess.QUEEN): 4,
    (chess.WHITE, chess.KING): 5,

    (chess.BLACK, chess.PAWN): 6,
    (chess.BLACK, chess.KNIGHT): 7,
    (chess.BLACK, chess.BISHOP): 8,
    (chess.BLACK, chess.ROOK): 9,
    (chess.BLACK, chess.QUEEN): 10,
    (chess.BLACK, chess.KING): 11,
}

INPUT_CHANNELS = 18


def encode_board(fen):
    board = chess.Board(fen)

    encoded = np.zeros(
        (INPUT_CHANNELS, 8, 8),
        dtype=np.float32
    )

    # Piece locations
    for square, piece in board.piece_map().items():
        channel = PIECE_TO_CHANNEL[
            (piece.color, piece.piece_type)
        ]

        row = 7 - chess.square_rank(square)
        col = chess.square_file(square)

        encoded[channel, row, col] = 1.0

    # Side to move
    if board.turn == chess.BLACK:
        encoded[12, :, :] = 1.0

    # Castling rights
    castling_checks = [
        board.has_kingside_castling_rights(chess.WHITE),
        board.has_queenside_castling_rights(chess.WHITE),
        board.has_kingside_castling_rights(chess.BLACK),
        board.has_queenside_castling_rights(chess.BLACK),
    ]

    for channel, has_right in enumerate(
        castling_checks,
        start=13
    ):
        if has_right:
            encoded[channel, :, :] = 1.0

    # En-passant square
    if board.ep_square is not None:
        row = 7 - chess.square_rank(board.ep_square)
        col = chess.square_file(board.ep_square)

        encoded[17, row, col] = 1.0

    return encoded

In [7]:
encoded_sample = encode_board(sample["fen"])

print("Encoded shape:", encoded_sample.shape)
assert encoded_sample.shape == (18, 8, 8)

Encoded shape: (18, 8, 8)


In [8]:
NUM_SQUARES = 64

BASE_MOVE_CLASSES = NUM_SQUARES * NUM_SQUARES
PROMOTION_OFFSET = BASE_MOVE_CLASSES

PROMOTION_PIECES = [
    chess.KNIGHT,
    chess.BISHOP,
    chess.ROOK,
    chess.QUEEN,
]

PROMOTION_TO_INDEX = {
    chess.KNIGHT: 0,
    chess.BISHOP: 1,
    chess.ROOK: 2,
    chess.QUEEN: 3,
}

NUM_MOVE_CLASSES = BASE_MOVE_CLASSES * 5


def encode_move(uci_move):
    move = chess.Move.from_uci(uci_move)

    base_index = (
        move.from_square * 64
        + move.to_square
    )

    if move.promotion is None:
        return base_index

    promotion_index = PROMOTION_TO_INDEX[move.promotion]

    return (
        PROMOTION_OFFSET
        + promotion_index * BASE_MOVE_CLASSES
        + base_index
    )


def decode_move(move_index):
    if not 0 <= move_index < NUM_MOVE_CLASSES:
        raise ValueError(f"Invalid move index: {move_index}")

    if move_index < BASE_MOVE_CLASSES:
        base_index = move_index
        promotion_piece = None

    else:
        promotion_index = (
            move_index - PROMOTION_OFFSET
        ) // BASE_MOVE_CLASSES

        base_index = (
            move_index - PROMOTION_OFFSET
        ) % BASE_MOVE_CLASSES

        promotion_piece = PROMOTION_PIECES[promotion_index]

    from_square = base_index // 64
    to_square = base_index % 64

    return chess.Move(
        from_square,
        to_square,
        promotion=promotion_piece
    )

In [9]:
# Verify ordinary moves and all promotion types.
test_moves = [
    "e2e4",
    "g1f3",
    "e7e8n",
    "e7e8b",
    "e7e8r",
    "e7e8q",
]

for uci in test_moves:
    decoded = decode_move(encode_move(uci)).uci()
    assert decoded == uci

print("Move encoding tests passed.")

Move encoding tests passed.


In [10]:
class ChessPolicyDataset(Dataset):

    def __init__(self, hf_split):
        self.data = hf_split

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):

        row = self.data[index]

        fen = row["fen"]
        target_move = row["target_move"]

        board = chess.Board(fen)

        if not board.is_valid():
            raise ValueError(f"Invalid board position: {fen}")

        legal_moves = list(board.legal_moves)

        if not legal_moves:
            raise ValueError(f"Terminal position: {fen}")

        legal_move_set = {
            move.uci() for move in legal_moves
        }

        target_index = encode_move(target_move)

        if target_move not in legal_move_set:
            raise ValueError(
                f"Illegal target {target_move} for {fen}"
            )

        # Extract candidate moves.
        raw_candidates = row["candidates"] or []

        candidate_moves = []

        for candidate in raw_candidates:
            if isinstance(candidate, dict):
                move = candidate["move"]
            else:
                move = candidate

            if move not in legal_move_set:
                raise ValueError(
                    f"Illegal candidate {move} for {fen}"
                )

            if move not in candidate_moves:
                candidate_moves.append(move)

        # Ensure the best target is always first.
        candidate_moves = [
            target_move
        ] + [
            move for move in candidate_moves
            if move != target_move
        ]

        if not candidate_moves:
            candidate_moves = [target_move]

        candidate_indices = [
            encode_move(move)
            for move in candidate_moves
        ]

        # Rank-based soft targets.
        # Lower rank number = higher target probability.
        ranks = np.arange(
            len(candidate_indices),
            dtype=np.float32
        )

        weights = np.exp(-ranks / 2.0)
        weights = weights / weights.sum()

        legal_indices = [
            encode_move(move.uci())
            for move in legal_moves
        ]

        return {
            "board": encode_board(fen),
            "target_index": target_index,
            "candidate_indices": candidate_indices,
            "candidate_weights": weights,
            "legal_indices": legal_indices,
        }

In [11]:
def chess_collate_fn(batch):

    batch_size = len(batch)

    boards = torch.tensor(
        np.stack([item["board"] for item in batch]),
        dtype=torch.float32
    )

    targets = torch.zeros(
        batch_size,
        NUM_MOVE_CLASSES,
        dtype=torch.float32
    )

    legal_masks = torch.zeros(
        batch_size,
        NUM_MOVE_CLASSES,
        dtype=torch.bool
    )

    target_indices = torch.tensor(
        [item["target_index"] for item in batch],
        dtype=torch.long
    )

    for i, item in enumerate(batch):

        candidate_indices = item["candidate_indices"]
        candidate_weights = item["candidate_weights"]

        targets[i, candidate_indices] = torch.tensor(
            candidate_weights,
            dtype=torch.float32
        )

        legal_masks[i, item["legal_indices"]] = True

    return boards, targets, target_indices, legal_masks

In [12]:
train_dataset = ChessPolicyDataset(dataset["train"])
val_dataset = ChessPolicyDataset(dataset["validation"])
test_dataset = ChessPolicyDataset(dataset["test"])

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
    collate_fn=chess_collate_fn
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
    collate_fn=chess_collate_fn
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
    collate_fn=chess_collate_fn
)

print("Train positions:", len(train_dataset))
print("Validation positions:", len(val_dataset))
print("Test positions:", len(test_dataset))

print("DataLoader workers:", train_loader.num_workers)

Train positions: 80000
Validation positions: 10000
Test positions: 10000
DataLoader workers: 0


In [13]:
class ResidualBlock(nn.Module):

    def __init__(self, channels):
        super().__init__()

        self.block = nn.Sequential(
            nn.Conv2d(
                channels,
                channels,
                kernel_size=3,
                padding=1,
                bias=False
            ),
            nn.BatchNorm2d(channels),
            nn.ReLU(inplace=True),

            nn.Conv2d(
                channels,
                channels,
                kernel_size=3,
                padding=1,
                bias=False
            ),
            nn.BatchNorm2d(channels)
        )

        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        residual = x
        x = self.block(x)
        x = x + residual
        return self.relu(x)

In [14]:
class ChessResNetPolicy(nn.Module):

    def __init__(
        self,
        input_channels=INPUT_CHANNELS,
        num_classes=NUM_MOVE_CLASSES,
        channels=128,
        num_blocks=6
    ):
        super().__init__()

        self.stem = nn.Sequential(
            nn.Conv2d(
                input_channels,
                channels,
                kernel_size=3,
                padding=1,
                bias=False
            ),
            nn.BatchNorm2d(channels),
            nn.ReLU(inplace=True)
        )

        self.residual_layers = nn.Sequential(
            *[
                ResidualBlock(channels)
                for _ in range(num_blocks)
            ]
        )

        self.policy_head = nn.Sequential(
            nn.Conv2d(
                channels,
                32,
                kernel_size=1,
                bias=False
            ),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),

            nn.Flatten(),

            nn.Linear(32 * 8 * 8, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.1),

            nn.Linear(512, num_classes)
        )

    def forward(self, x):

        x = self.stem(x)
        x = self.residual_layers(x)
        x = self.policy_head(x)

        return x

In [15]:
model = ChessResNetPolicy().to(DEVICE)

total_params = sum(
    parameter.numel()
    for parameter in model.parameters()
)

print(model)
print(f"\nTotal parameters: {total_params:,}")
print(f"Parameters in millions: {total_params / 1e6:.2f}M")

ChessResNetPolicy(
  (stem): Sequential(
    (0): Conv2d(18, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (1): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (2): ReLU(inplace=True)
  )
  (residual_layers): Sequential(
    (0): ResidualBlock(
      (block): Sequential(
        (0): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (1): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
        (2): ReLU(inplace=True)
        (3): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (4): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      )
      (relu): ReLU(inplace=True)
    )
    (1): ResidualBlock(
      (block): Sequential(
        (0): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (1): BatchNorm2d(128, eps=1

In [16]:
def apply_legal_move_mask(logits, legal_masks):

    if not legal_masks.any(dim=1).all():
        raise ValueError("A position has no legal moves.")

    return logits.masked_fill(
        ~legal_masks,
        -1e9
    )


def policy_loss(logits, target_probs, legal_masks):

    masked_logits = apply_legal_move_mask(
        logits,
        legal_masks
    )

    log_probs = F.log_softmax(
        masked_logits,
        dim=1
    )

    loss = -(
        target_probs * log_probs
    ).sum(dim=1).mean()

    return loss

In [17]:
def evaluate_model(model, loader):

    model.eval()

    total_loss = 0.0
    total_samples = 0

    hits = {
        1: 0,
        3: 0,
        5: 0,
        10: 0
    }

    with torch.no_grad():

        for boards, targets, target_indices, legal_masks in loader:

            boards = boards.to(DEVICE)
            targets = targets.to(DEVICE)
            target_indices = target_indices.to(DEVICE)
            legal_masks = legal_masks.to(DEVICE)

            logits = model(boards)

            loss = policy_loss(
                logits,
                targets,
                legal_masks
            )

            masked_logits = apply_legal_move_mask(
                logits,
                legal_masks
            )

            batch_size = boards.size(0)

            total_loss += loss.item() * batch_size
            total_samples += batch_size

            for k in hits:

                predictions = torch.topk(
                    masked_logits,
                    k=k,
                    dim=1
                ).indices

                correct = (
                    predictions == target_indices.unsqueeze(1)
                ).any(dim=1)

                hits[k] += correct.sum().item()

    return {
        "loss": total_loss / total_samples,
        "top1": hits[1] / total_samples,
        "top3": hits[3] / total_samples,
        "top5": hits[5] / total_samples,
        "top10": hits[10] / total_samples
    }

In [18]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=2
)


In [19]:
best_val_loss = float("inf")
best_state = None

epochs_without_improvement = 0
history = []

for epoch in range(EPOCHS):

    model.train()

    running_loss = 0.0
    total_samples = 0

    progress = tqdm(
        train_loader,
        desc=f"Epoch {epoch + 1}/{EPOCHS}"
    )

    for boards, targets, target_indices, legal_masks in progress:

        boards = boards.to(DEVICE)
        targets = targets.to(DEVICE)
        legal_masks = legal_masks.to(DEVICE)

        optimizer.zero_grad(set_to_none=True)

        logits = model(boards)

        loss = policy_loss(
            logits,
            targets,
            legal_masks
        )

        loss.backward()

        optimizer.step()

        batch_size = boards.size(0)

        running_loss += loss.item() * batch_size
        total_samples += batch_size

        progress.set_postfix(
            loss=f"{loss.item():.4f}"
        )

    train_loss = running_loss / total_samples

    val_metrics = evaluate_model(
        model,
        val_loader
    )

    scheduler.step(val_metrics["loss"])

    print(
        f"\nEpoch {epoch + 1}: "
        f"train_loss={train_loss:.4f}, "
        f"val_loss={val_metrics['loss']:.4f}, "
        f"top1={val_metrics['top1']:.2%}, "
        f"top5={val_metrics['top5']:.2%}, "
        f"top10={val_metrics['top10']:.2%}"
    )

    history.append({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        **{
            f"val_{key}": value
            for key, value in val_metrics.items()
        }
    })

    current_val_loss = val_metrics["loss"]

    if current_val_loss < best_val_loss - MIN_DELTA:

        best_val_loss = current_val_loss

        best_state = deepcopy(model.state_dict())

        epochs_without_improvement = 0

        torch.save(
            best_state,
            "best_chess_policy_state.pt"
        )

        print("Saved improved checkpoint.")

    else:

        epochs_without_improvement += 1

        print(
            "No improvement:",
            epochs_without_improvement,
            "/",
            PATIENCE
        )

    if epochs_without_improvement >= PATIENCE:

        print("Early stopping triggered.")
        break

if best_state is None:
    raise RuntimeError("Training did not produce a valid checkpoint.")

model.load_state_dict(best_state)

print("\nBest validation loss:", best_val_loss)

Epoch 1/20:   0%|          | 0/625 [00:00<?, ?it/s]


Epoch 1: train_loss=2.9541, val_loss=2.9780, top1=14.52%, top5=43.83%, top10=61.58%
Saved improved checkpoint.


Epoch 2/20:   0%|          | 0/625 [00:00<?, ?it/s]


Epoch 2: train_loss=2.9038, val_loss=2.9487, top1=15.67%, top5=46.40%, top10=64.43%
Saved improved checkpoint.


Epoch 3/20:   0%|          | 0/625 [00:00<?, ?it/s]


Epoch 3: train_loss=2.8408, val_loss=2.9290, top1=16.97%, top5=47.52%, top10=65.20%
Saved improved checkpoint.


... (6 outputs left out)

Epoch 7/20:   0%|          | 0/625 [00:00<?, ?it/s]


Epoch 7: train_loss=2.5091, val_loss=2.9618, top1=17.31%, top5=49.50%, top10=66.89%
No improvement: 3 / 4


Epoch 8/20:   0%|          | 0/625 [00:00<?, ?it/s]


Epoch 8: train_loss=2.3882, val_loss=3.0032, top1=17.70%, top5=49.97%, top10=67.42%
No improvement: 4 / 4
Early stopping triggered.

Best validation loss: 2.917908066177368


In [20]:
test_metrics = evaluate_model(
    model,
    test_loader
)

print("Final Test Results")
print("------------------")

for metric, value in test_metrics.items():
    if metric == "loss":
        print(f"{metric}: {value:.4f}")
    else:
        print(f"{metric}: {value:.2%}")

history_df = pd.DataFrame(history)
display(history_df)

Final Test Results
------------------
loss: 2.8778
top1: 17.34%
top3: 36.93%
top5: 48.47%
top10: 66.42%


,epoch,train_loss,val_loss,val_top1,val_top3,val_top5,val_top10
0,1,2.954117,2.977955,0.1452,0.3269,0.4383,0.6158
1,2,2.903783,2.948727,0.1567,0.3416,0.4640,0.6443
2,3,2.840794,2.929035,0.1697,0.3595,0.4752,0.6520
3,4,2.764722,2.917908,0.1709,0.3673,0.4871,0.6646
4,5,2.679482,2.934193,0.1736,0.3712,0.4907,0.6720
5,6,2.593053,2.947977,0.1719,0.3752,0.4980,0.6763
6,7,2.509106,2.961780,0.1731,0.3742,0.4950,0.6689
7,8,2.388171,3.003190,0.1770,0.3809,0.4997,0.6742


In [21]:
def get_candidate_moves(
    model,
    fen,
    top_k=5
):

    model.eval()

    board = chess.Board(fen)

    if not board.is_valid():
        raise ValueError("Invalid FEN.")

    legal_moves = list(board.legal_moves)

    if not legal_moves:
        return []

    encoded = encode_board(fen)

    tensor = torch.tensor(
        encoded,
        dtype=torch.float32
    ).unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        logits = model(tensor)[0]

    legal_indices = [
        encode_move(move.uci())
        for move in legal_moves
    ]

    legal_indices_tensor = torch.tensor(
        legal_indices,
        dtype=torch.long,
        device=DEVICE
    )

    legal_scores = logits[legal_indices_tensor]

    k = min(top_k, len(legal_moves))

    top_scores, top_positions = torch.topk(
        legal_scores,
        k=k
    )

    results = []

    for score, position in zip(
        top_scores.tolist(),
        top_positions.tolist()
    ):
        move = legal_moves[position]

        results.append({
            "move": move.uci(),
            "score": score
        })

    return results

In [24]:
def inspect_prediction(model, row, top_k=5):

    board = chess.Board(row["fen"])

    candidates = get_candidate_moves(
        model,
        row["fen"],
        top_k
    )

    target_move = row["target_move"]

    predicted_moves = [
        item["move"] for item in candidates
    ]

    target_rank = (
        predicted_moves.index(target_move) + 1
        if target_move in predicted_moves
        else None
    )

    print(board)

    print("\nStockfish best candidate:", target_move)
    print("\nPolicy candidates:")

    for rank, candidate in enumerate(candidates, 1):
        print(
            f"{rank}. {candidate['move']} "
            f"(logit={candidate['score']:.4f})"
        )

    print("\nStockfish move rank:", target_rank or "Outside top-K")

    return target_rank


In [25]:
random.seed(42)

sample_indices = random.sample(
    range(len(dataset["test"])),
    min(10, len(dataset["test"]))
)

for i, index in enumerate(sample_indices, 1):

    print(f"\n{'=' * 15} Test position {i} {'=' * 15}")

    inspect_prediction(
        model,
        dataset["test"][index],
        top_k=5
    )


=============== Test position 1 ===============
r n . q k b n r
p p p . p p p .
. . . . . . . .
. . . . . b P p
. . . P p P . .
P . N . B . . .
. P P . . . . P
R . . Q K B N R

Stockfish best candidate: e7e6

Policy candidates:
1. e7e6 (logit=0.7208)
2. c7c6 (logit=0.0593)
3. g8f6 (logit=-0.1008)
4. e7e5 (logit=-0.3196)
5. b8c6 (logit=-0.3793)

Stockfish move rank: 1

=============== Test position 2 ===============
. . . . . . . .
. . . . . . . .
. . . . . . k .
. . . . . . . .
. . . p b . . p
. . . . . . . .
. . . . . K . .
. . . . . . . .

Stockfish best candidate: d4d3

Policy candidates:
1. h4h3 (logit=0.9821)
2. d4d3 (logit=0.3626)
3. g6f5 (logit=0.1141)
4. g6g5 (logit=-0.0200)
5. g6f7 (logit=-0.0633)

Stockfish move rank: 2

=============== Test position 3 ===============
r n b q . r k .
p p . p n p p p
. . p . p . . .
. . . . P . . .
. b . P . . . .
. . N . . . . .
P P P . B P P P
R . B Q K . N R

Stockfish best candidate: e2d3

Policy candidates:
1. g1f3 (logit=0.8601)
2. a2a3

In [26]:
def evaluate_test_ranking(model, test_split, top_k=10):

    ranks = []

    for row in tqdm(test_split, desc="Evaluating test positions"):

        candidates = get_candidate_moves(
            model,
            row["fen"],
            top_k=top_k
        )

        predicted_moves = [
            item["move"] for item in candidates
        ]

        target = row["target_move"]

        if target in predicted_moves:
            rank = predicted_moves.index(target) + 1
        else:
            rank = top_k + 1

        ranks.append(rank)

    ranks = np.array(ranks)

    results = {
        "positions": len(ranks),
        "top1": np.mean(ranks <= 1),
        "top3": np.mean(ranks <= 3),
        "top5": np.mean(ranks <= 5),
        "top10": np.mean(ranks <= 10),
        "mean_rank_capped": np.mean(ranks),
        "mrr": np.mean(1.0 / ranks)
    }

    print("\nFull Test Ranking Evaluation")
    print("----------------------------")

    for key, value in results.items():

        if key == "positions":
            print(f"{key}: {value}")
        else:
            print(f"{key}: {value:.4f} ({value:.2%})")

    return results


full_test_results = evaluate_test_ranking(
    model,
    dataset["test"],
    top_k=10
)

Evaluating test positions:   0%|          | 0/10000 [00:00<?, ?it/s]


Full Test Ranking Evaluation
----------------------------
positions: 10000
top1: 0.1734 (17.34%)
top3: 0.3693 (36.93%)
top5: 0.4847 (48.47%)
top10: 0.6642 (66.42%)
mean_rank_capped: 6.2668 (626.68%)
mrr: 0.3382 (33.82%)


In [ ]:
MODEL_DIR = Path("chessmind_policy_checkpoint")
MODEL_DIR.mkdir(exist_ok=True)

config = {
    "architecture": "ChessResNetPolicy",
    "input_channels": INPUT_CHANNELS,
    "num_classes": NUM_MOVE_CLASSES,
    "channels": 128,
    "num_blocks": 6,
    "move_encoding": "from_square_to_square_with_promotion",
    "board_encoding": "18_channel",
    "candidate_supervision": "rank_weighted",
    "rank_temperature": 2.0,
    "dataset": HF_DATASET_ID,
}

checkpoint = {
    "model_state_dict": {
        key: value.detach().cpu()
        for key, value in model.state_dict().items()
    },
    "config": config,
    "metrics": test_metrics,
    "best_validation_loss": best_val_loss,
}

torch.save(
    checkpoint,
    MODEL_DIR / "model_checkpoint.pth"
)

with open(MODEL_DIR / "config.json", "w") as f:
    json.dump(config, f, indent=2)

with open(MODEL_DIR / "metrics.json", "w") as f:
    json.dump(test_metrics, f, indent=2)

print("Saved model files:")

for path in MODEL_DIR.iterdir():
    print(
        path.name,
        f"{path.stat().st_size / 1024 / 1024:.2f} MB"
    )

Saved files:
chessmind_policy_checkpoint/config.json (0.00 MB)
chessmind_policy_checkpoint/model_checkpoint.pth (49.25 MB)


In [ ]:
REPO_ID = "Mridul-Dixit/chessmind-arena-policy-model"

api = HfApi(token=HF_TOKEN)

# Create the repository only if it does not already exist.
api.create_repo(
    repo_id=REPO_ID,
    repo_type="model",
    exist_ok=True,
    private=False
)

# Upload to the same repository.
api.upload_folder(
    repo_id=REPO_ID,
    repo_type="model",
    folder_path=str(MODEL_DIR),
    commit_message="Update ChessMind Arena ResNet policy model"
)

print(
    f"Model uploaded successfully: "
    f"https://huggingface.co/{REPO_ID}"
)

Uploaded checkpoint to https://huggingface.co/Mridul-Dixit/chessmind-arena-policy-model
